
### Compare LR, PCA, RF with TemporalVAE on Mouse atlas data, K-Fold test
### for Figure3 C

In [1]:

# -*-coding:utf-8 -*-
import os
if os.getcwd().split("/")[-1] != "TemporalVAE":
    os.chdir("..")
import sys
sys.path.append(os.getcwd())

import anndata as ad
import pandas as pd


# Linear regression

In [2]:

from collections import Counter
from utils.utils_DandanProject import *
from utils.utils_Dandan_plot import *
import time
import logging
from utils.logging_system import LogHelper
from sklearn.linear_model import LinearRegression
def corr(x1, x2, special_str=""):
    from scipy.stats import spearmanr, kendalltau
    sp_correlation, sp_p_value = spearmanr(x1, x2)
    ke_correlation, ke_p_value = kendalltau(x1, x2)

    sp = f"{special_str} spearman correlation score: {sp_correlation}, p-value: {sp_p_value}."
    print(sp)
    ke = f"{special_str} kendalltau correlation score: {ke_correlation},p-value: {ke_p_value}."
    print(ke)

    return sp, ke
def LR():
    method = "linearRegression"
    save_path=f"results/240509_kFold_mouse_atlas_data_onlyTestTime/{method}_atlas/"
    # save_path=f"/mnt/yijun/nfs_share/awa_project/pairsRegulatePrediction/GPLVM_dandan/results/240322_forFig3_compareWithBaseLine/{method}_atlas/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    data_golbal_path = "data/"
    data_path = "/mouse_embryonic_development/preprocess_adata_JAX_dataset_combine_minGene100_minCell50_hvg1000/"
    sc_data_file_csv = f"{data_path}/data_count_hvg.csv"
    cell_info_file_csv = f"{data_path}/cell_with_time.csv"

    # ---------------------------------------set logger and parameters, creat result save path and folder----------------------------------------------
    logger_file = f'{save_path}/{method}_run.log'
    LogHelper.setup(log_path=logger_file, level='INFO')
    _logger = logging.getLogger(__name__)
    _logger.info("Finished setting up the logger at: {}.".format(logger_file))
    _logger.info("Train on dataset: {}.".format(data_golbal_path + data_path))

    sc_expression_df, cell_time = preprocessData_and_dropout_some_donor_or_gene(data_golbal_path, sc_data_file_csv, cell_info_file_csv,
                                                                                min_cell_num=50,
                                                                                min_gene_num=100)
    # ---------------------------------------- set donor list and dictionary -----------------------------------------------------
    donor_list = np.unique(cell_time["donor"])
    donor_list = sorted(donor_list, key=Embryodonor_resort_key)
    donor_dic = dict()
    for i in range(len(donor_list)):
        donor_dic[donor_list[i]] = i
    batch_dic = donor_dic.copy()
    print("Consider donor as batch effect, donor use label: {}".format(donor_dic))
    print("For each donor (donor_id, cell_num):{} ".format(Counter(cell_time["donor"])))

    kFold_test_result_df = pd.DataFrame(columns=['time', 'pseudotime'])

    # use one donor as test set, other as train set
    adata = ad.AnnData(X=sc_expression_df,obs=cell_time)
    print(len(donor_list))


    for donor in donor_list:
        # donor="embryo_1"
        # start_time = time.time()

        train_adata = adata[adata.obs["donor"] != donor].copy()
        test_adata = adata[adata.obs["donor"] == donor].copy()
        # 初始化线性回归模型
        model = LinearRegression()
        # 训练模型
        model.fit(train_adata.X, train_adata.obs["time"])

        # 使用模型进行预测
        predictions = model.predict(test_adata.X)
        test_result_df = pd.DataFrame(test_adata.obs["time"])
        test_result_df["pseudotime"] = predictions
        # test_result_df["pseudotime"] = classifier.predict(test_lowDim)

        kFold_test_result_df = pd.concat([kFold_test_result_df, test_result_df], axis=0)
        # end_time = time.time()
        # elapsed_time = end_time - start_time
        # _logger.info(f"**** Total execution time: {elapsed_time} seconds **** k-fold on {donor}")
        # exit(1)

    print("k-fold test final result:")
    corr(kFold_test_result_df["time"], kFold_test_result_df["pseudotime"])
    kFold_test_result_df.to_csv(f'{save_path}/result_df.csv', index=True)
    print(f"test result save at {save_path}/result_df.csv")

    f = plot_psupertime_density(kFold_test_result_df, label_key="time", psupertime_key="pseudotime")
    f.savefig(f"{save_path}/labelsOverPsupertime.png")
    print(f"figure save at {save_path}/labelsOverPsupertime.png")

In [ ]:

LR()



2024-05-09 10:25:34,831 INFO - __main__ - Finished setting up the logger at: results/240509_kFold_mouse_atlas_data_onlyTestTime/linearRegression_atlas//linearRegression_run.log. 
2024-05-09 10:25:34,974 INFO - __main__ - Train on dataset: data//mouse_embryonic_development/preprocess_adata_JAX_dataset_combine_minGene100_minCell50_hvg1000/. 
2024-05-09 10:25:34,974 INFO - utils.utils_DandanProject - the original sc expression anndata should be gene as row, cell as column 
2024-05-09 10:27:36,859 INFO - utils.utils_DandanProject - read the original sc expression anndata with shape (gene, cell): (979, 881168) 
2024-05-09 10:27:36,871 INFO - utils.utils_DandanProject - Import data, cell number: 881168, gene number: 979 
2024-05-09 10:27:48,451 INFO - utils.utils_DandanProject - After drop gene threshold: 50, cell threshold: 100, remain adata shape: (881168, 979) 
2024-05-09 10:27:48,451 INFO - utils.utils_DandanProject - Drop cells with less than 100 gene expression, drop genes which none e

Consider donor as batch effect, donor use label: {'embryo_1': 0, 'embryo_2': 1, 'embryo_3': 2, 'embryo_4': 3, 'embryo_5': 4, 'embryo_6': 5, 'embryo_7': 6, 'embryo_8': 7, 'embryo_9': 8, 'embryo_10': 9, 'embryo_11': 10, 'embryo_12': 11, 'embryo_13': 12, 'embryo_14': 13, 'embryo_15': 14, 'embryo_16': 15, 'embryo_17': 16, 'embryo_18': 17, 'embryo_19': 18, 'embryo_20': 19, 'embryo_21': 20, 'embryo_22': 21, 'embryo_23': 22, 'embryo_24': 23, 'embryo_25': 24, 'embryo_26': 25, 'embryo_27': 26, 'embryo_28': 27, 'embryo_29': 28, 'embryo_30': 29, 'embryo_31': 30, 'embryo_32': 31, 'embryo_33': 32, 'embryo_34': 33, 'embryo_35': 34, 'embryo_36': 35, 'embryo_37': 36, 'embryo_38': 37, 'embryo_39': 38, 'embryo_40': 39, 'embryo_41': 40, 'embryo_42': 41, 'embryo_43': 42, 'embryo_44': 43, 'embryo_45': 44, 'embryo_46': 45, 'embryo_47': 46, 'embryo_48': 47, 'embryo_49': 48, 'embryo_50': 49, 'embryo_51': 50, 'embryo_52': 51, 'embryo_53': 52, 'embryo_54': 53, 'embryo_55': 54, 'embryo_56': 55, 'embryo_57': 56, 

/mnt/yijun/nfs_share/yijun_tmp/ipykernel_3919664/2842430994.py:53: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(X=sc_expression_df,obs=cell_time)


72



# PCA

In [ ]:

from sklearn.decomposition import PCA
def PCA():
    method = "PCA"
    save_path=f"/mnt/yijun/nfs_share/awa_project/pairsRegulatePrediction/GPLVM_dandan/results/240328_kFold_mouse_atlas_data_onlyTestTime/{method}_atlas/"
    # save_path=f"/mnt/yijun/nfs_share/awa_project/pairsRegulatePrediction/GPLVM_dandan/results/240322_forFig3_compareWithBaseLine/{method}_atlas/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    data_golbal_path = "/mnt/yijun/nfs_share/awa_project/pairsRegulatePrediction/GPLVM_dandan/data/"
    data_path = "/mouse_embryonic_development/preprocess_adata_JAX_dataset_combine_minGene100_minCell50_hvg1000/"
    sc_data_file_csv = f"{data_path}/data_count_hvg.csv"
    cell_info_file_csv = f"{data_path}/cell_with_time.csv"

    # ---------------------------------------set logger and parameters, creat result save path and folder----------------------------------------------
    logger_file = f'{save_path}/{method}_run.log'
    LogHelper.setup(log_path=logger_file, level='INFO')
    _logger = logging.getLogger(__name__)
    _logger.info("Finished setting up the logger at: {}.".format(logger_file))
    _logger.info("Train on dataset: {}.".format(data_golbal_path + data_path))

    sc_expression_df, cell_time = preprocessData_and_dropout_some_donor_or_gene(data_golbal_path, sc_data_file_csv, cell_info_file_csv,
                                                                                min_cell_num=50,
                                                                                min_gene_num=100)
    # ---------------------------------------- set donor list and dictionary -----------------------------------------------------
    donor_list = np.unique(cell_time["donor"])
    donor_list = sorted(donor_list, key=Embryodonor_resort_key)
    donor_dic = dict()
    for i in range(len(donor_list)):
        donor_dic[donor_list[i]] = i
    batch_dic = donor_dic.copy()
    print("Consider donor as batch effect, donor use label: {}".format(donor_dic))
    print("For each donor (donor_id, cell_num):{} ".format(Counter(cell_time["donor"])))

    kFold_test_result_df = pd.DataFrame(columns=['time', 'pseudotime'])

    # use one donor as test set, other as train set
    adata = ad.AnnData(X=sc_expression_df,obs=cell_time)
    print(len(donor_list))
    for donor in donor_list:
        # donor = "embryo_1"
        # start_time = time.time()

        train_adata = adata[adata.obs["donor"] != donor].copy()
        test_adata = adata[adata.obs["donor"] == donor].copy()
        # initiate PCA and classifier
        pca = PCA(n_components=2)
        # classifier = DecisionTreeClassifier()
        # transform / fit
        train_lowDim = pca.fit_transform(train_adata.X)
        # classifier.fit(train_lowDim, train_adata.obs["time"])
        # predict "new" data
        test_lowDim = pca.transform(test_adata.X)
        # predict labels using the trained classifier
        test_result_df = pd.DataFrame(test_adata.obs["time"])
        test_result_df["pseudotime"] = test_lowDim[:, 0]
        # test_result_df["pseudotime"] = classifier.predict(test_lowDim)

        kFold_test_result_df = pd.concat([kFold_test_result_df, test_result_df], axis=0)
        # end_time = time.time()
        # elapsed_time = end_time - start_time
        # _logger.info(f"**** Total execution time: {elapsed_time} seconds **** k-fold on {donor}")
        # exit(1)
    print("k-fold test final result:")
    corr(kFold_test_result_df["time"], kFold_test_result_df["pseudotime"])
    kFold_test_result_df.to_csv(f'{save_path}/result_df.csv', index=True)
    print(f"test result save at {save_path}/result_df.csv")

    f = plot_psupertime_density(kFold_test_result_df, label_key="time", psupertime_key="pseudotime")
    f.savefig(f"{save_path}/labelsOverPsupertime.png")
    print(f"figure save at {save_path}/labelsOverPsupertime.png")

In [ ]:

PCA()


# RF

In [ ]:

def RF():
    method = "randomForest"
    save_path=f"/mnt/yijun/nfs_share/awa_project/pairsRegulatePrediction/GPLVM_dandan/results/240328_kFold_mouse_atlas_data_onlyTestTime/{method}_atlas/"
    # save_path=f"/mnt/yijun/nfs_share/awa_project/pairsRegulatePrediction/GPLVM_dandan/results/240322_forFig3_compareWithBaseLine/{method}_atlas/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    data_golbal_path = "/mnt/yijun/nfs_share/awa_project/pairsRegulatePrediction/GPLVM_dandan/data/"
    data_path = "/mouse_embryonic_development/preprocess_adata_JAX_dataset_combine_minGene100_minCell50_hvg1000/"
    sc_data_file_csv = f"{data_path}/data_count_hvg.csv"
    cell_info_file_csv = f"{data_path}/cell_with_time.csv"

    # ---------------------------------------set logger and parameters, creat result save path and folder----------------------------------------------
    logger_file = f'{save_path}/{method}_run.log'
    LogHelper.setup(log_path=logger_file, level='INFO')
    _logger = logging.getLogger(__name__)
    _logger.info("Finished setting up the logger at: {}.".format(logger_file))
    _logger.info("Train on dataset: {}.".format(data_golbal_path + data_path))

    sc_expression_df, cell_time = preprocessData_and_dropout_some_donor_or_gene(data_golbal_path, sc_data_file_csv, cell_info_file_csv,
                                                                                min_cell_num=50,
                                                                                min_gene_num=100)
    # ---------------------------------------- set donor list and dictionary -----------------------------------------------------
    donor_list = np.unique(cell_time["donor"])
    donor_list = sorted(donor_list, key=Embryodonor_resort_key)
    donor_dic = dict()
    for i in range(len(donor_list)):
        donor_dic[donor_list[i]] = i
    # batch_dic = donor_dic.copy()
    print("Consider donor as batch effect, donor use label: {}".format(donor_dic))
    print("For each donor (donor_id, cell_num):{} ".format(Counter(cell_time["donor"])))

    kFold_test_result_df = pd.DataFrame(columns=['time', 'pseudotime'])

    # use one donor as test set, other as train set
    adata = ad.AnnData(X=sc_expression_df,obs=cell_time)
    print(len(donor_list))
    for donor in donor_list:
        # donor = "embryo_1"
        # start_time = time.time()
        train_adata = adata[adata.obs["donor"] != donor].copy()
        test_adata = adata[adata.obs["donor"] == donor].copy()
        # move one cell from test to train to occupy the test time
        RF_model = random_forest_regressor(train_x=train_adata.X, train_y=train_adata.obs["time"])
        # RF_model = random_forest_classifier(train_x=train_adata.X, train_y=train_adata.obs["time"])
        test_y_predicted = RF_model.predict(test_adata.X)

        test_result_df = pd.DataFrame(test_adata.obs["time"],index=test_adata.obs.index)
        test_result_df["pseudotime"] = test_y_predicted

        kFold_test_result_df = pd.concat([kFold_test_result_df, test_result_df], axis=0)
        # end_time = time.time()
        # elapsed_time = end_time - start_time
        # _logger.info(f"**** Total execution time: {elapsed_time} seconds **** k-fold on {donor}")
        # exit(1)
    print("k-fold test final result:")
    corr(kFold_test_result_df["time"], kFold_test_result_df["pseudotime"])
    kFold_test_result_df.to_csv(f'{save_path}/result_df.csv', index=True)
    print(f"test result save at {save_path}/result_df.csv")

    f = plot_psupertime_density(kFold_test_result_df, label_key="time", psupertime_key="pseudotime")
    f.savefig(f"{save_path}/labelsOverPsupertime.png")
    print(f"figure save at {save_path}/labelsOverPsupertime.png")
def random_forest_regressor(train_x, train_y):
    from sklearn.ensemble import RandomForestRegressor
    model = RandomForestRegressor(max_depth=2, random_state=0)
    model.fit(train_x, train_y)
    return model

In [ ]:

RF()


# read TemporalVAE result, and compare with baseline methods

In [ ]:

from utils.utils_Dandan_plot import calculate_real_predict_corrlation_score
import pandas as pd
# ----------------------- k-fold on mouse atlas data, for temporalVAE compare with LR, PCA, RF -----------------------
file_name = "results/230827_trainOn_mouse_embryonic_development_kFold_testOnYZdata0809/mouse_embryonic_development/preprocess_adata_JAX_dataset_combine_minGene100_minCell50_hvg1000/supervise_vae_regressionclfdecoder_dim50_timeembryoneg5to5_epoch100_dropDonorno_mouseEmbryonicDevelopment_embryoneg5to5/result_df.csv"
data_pd = pd.read_csv(file_name)
VAE = calculate_real_predict_corrlation_score(data_pd["time"], data_pd["predicted_time"])

file_name = "/mnt/yijun/nfs_share/awa_project/pairsRegulatePrediction/GPLVM_dandan/results/240322_forFig3_compareWithBaseLine/linearRegression_atlas/result_df.csv"
data_pd = pd.read_csv(file_name)
LR = calculate_real_predict_corrlation_score(data_pd["time"], data_pd["pseudotime"])

file_name = "/mnt/yijun/nfs_share/awa_project/pairsRegulatePrediction/GPLVM_dandan/results/240322_forFig3_compareWithBaseLine/PCA_atlas/result_df.csv"
data_pd = pd.read_csv(file_name)
PCA = calculate_real_predict_corrlation_score(data_pd["time"], data_pd["pseudotime"])

file_name = "/mnt/yijun/nfs_share/awa_project/pairsRegulatePrediction/GPLVM_dandan/results/240322_forFig3_compareWithBaseLine/randomForest_atlas/result_df.csv"
data_pd = pd.read_csv(file_name)
RF = calculate_real_predict_corrlation_score(data_pd["time"], data_pd["pseudotime"])
data = {
    'Method': ['TemporalVAE', 'TemporalVAE', 'LR', 'LR', 'PCA', 'PCA', "RF", "RF"],
    'Correlation Type': ['Spearman', 'Pearson', 'Spearman', 'Pearson', 'Spearman', 'Pearson', 'Spearman', 'Pearson'],
    'Value': [0.89082, 0.91370, 0.78595, 0.79398, 0.25161, 0.14779, 0.07168, 0.36478]
}

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# for "embryo_1"
# time_dic={"TemporalVAE":2161.1639816761017,}
df = pd.DataFrame(data)

# 设置绘图风格
sns.set(style="whitegrid")

# 创建条形图
plt.figure(figsize=(8, 6))
barplot = sns.barplot(x='Method', y='Value', hue='Correlation Type', data=df, palette=["#B0E0E6", "#D8BFD8"])

# 添加标题和坐标轴标签
plt.title('Method Performance: Spearman vs Pearson Correlation', fontsize=16)
plt.ylabel('Correlation Value', fontsize=14)
plt.xlabel('Method', fontsize=14)

# 调整图例
plt.legend(title='Correlation Type', title_fontsize='13', fontsize='12')

# 在每个条形上显示数值
for p in barplot.patches:
    barplot.annotate(format(p.get_height(), '.3f'),
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center',
                     xytext=(0, 10),
                     textcoords='offset points')

# 设置Y轴范围以清晰展示负相关性，添加Y=0参考线强调正负相关性
plt.ylim(0, 1)
plt.axhline(0, color='black', linewidth=1, linestyle='--')

# 美化图表
sns.despine(offset=10, trim=True)  # 减少边框
plt.tight_layout()  # 自动调整子图参数,使之填充整个图像区域

# 显示图表
plt.show()